# Employee Sentiment Analysis

## Project Overview

This notebook performs a comprehensive analysis of employee email messages to assess **sentiment and engagement**. We work with an unlabeled dataset of employee messages and derive actionable insights using Natural Language Processing (NLP) and statistical analysis techniques.

### Tasks Covered:
1. **Sentiment Labeling** - Classify each message as Positive, Negative, or Neutral
2. **Exploratory Data Analysis (EDA)** - Understand data structure, distributions, and trends
3. **Monthly Sentiment Scoring** - Compute monthly scores per employee
4. **Employee Ranking** - Top 3 positive and negative employees per month
5. **Flight Risk Identification** - Flag employees with 4+ negative messages in 30 days
6. **Predictive Modeling** - Linear regression to analyze sentiment trends

### Libraries Used:
- **TextBlob** for NLP sentiment analysis
- **scikit-learn** for machine learning (Linear Regression)
- **pandas, numpy** for data manipulation
- **matplotlib, seaborn** for visualization
- **wordcloud** for word cloud generation

---
## Setup and Imports

In [ ]:
# Standard library imports
import os
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# NLP - Sentiment Analysis
from textblob import TextBlob

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Set visualization style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Create output directory
os.makedirs('visualizations', exist_ok=True)

print('All libraries loaded successfully!')

---
## Step 0: Data Loading and Preprocessing

We begin by loading the raw dataset (`test.csv`) and performing basic data cleaning:
- Parse date columns
- Handle missing values
- Standardize employee email addresses

In [ ]:
# Load the dataset
df = pd.read_csv('test.csv')

print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'\nColumn Names: {df.columns.tolist()}')
print(f'\nData Types:')
print(df.dtypes)

In [ ]:
# Preview the first 5 rows
df.head()

In [ ]:
# Check for missing values
print('Missing Values:')
print(df.isnull().sum())
print(f'\nTotal missing cells: {df.isnull().sum().sum()}')

In [ ]:
# Parse dates and clean data
df['date'] = pd.to_datetime(df['date'], format='mixed', dayfirst=False)
df['from'] = df['from'].str.strip().str.lower()

print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Total unique employees: {df["from"].nunique()}')
print(f'Total messages: {len(df)}')

**Observation:** The dataset contains employee email messages with Subject, body, date, and sender (from) columns. Dates span multiple years and there are several unique employees. Some messages may have missing Subject or body fields which we'll handle during sentiment analysis.

---
## Task 1: Sentiment Labeling

### Approach
We use **TextBlob**, a popular NLP library built on top of NLTK, for sentiment analysis. TextBlob provides a polarity score ranging from -1.0 (most negative) to +1.0 (most positive).

### Classification Thresholds:
- **Polarity > 0.1** --> Positive
- **Polarity < -0.1** --> Negative  
- **Otherwise** --> Neutral

### Why TextBlob?
- Runs locally without API keys
- Well-established and reproducible
- Handles general English text well
- Combines Subject + Body for richer context

In [ ]:
def get_sentiment_label(text):
    """
    Analyze sentiment using TextBlob.
    Returns: (sentiment_label, polarity_score)
    """
    if pd.isna(text) or str(text).strip() == '':
        return 'Neutral', 0.0
    
    try:
        blob = TextBlob(str(text))
        polarity = blob.sentiment.polarity
        
        if polarity > 0.1:
            return 'Positive', polarity
        elif polarity < -0.1:
            return 'Negative', polarity
        else:
            return 'Neutral', polarity
    except Exception:
        return 'Neutral', 0.0

# Combine subject and body for richer context
df['full_text'] = df['Subject'].fillna('') + ' ' + df['body'].fillna('')

# Apply sentiment analysis
print('Applying sentiment analysis... (this may take a minute)')
results = df['full_text'].apply(lambda x: get_sentiment_label(x))
df['sentiment'] = results.apply(lambda x: x[0])
df['polarity'] = results.apply(lambda x: x[1])

print('Sentiment labeling complete!')

In [ ]:
# Display sentiment distribution
print('Sentiment Distribution:')
print('=' * 40)
sentiment_counts = df['sentiment'].value_counts()
for sentiment, count in sentiment_counts.items():
    pct = count / len(df) * 100
    print(f'  {sentiment:>10}: {count:>5} messages ({pct:.1f}%)')

print(f'\nAverage Polarity Score: {df["polarity"].mean():.4f}')
print(f'Std Polarity Score: {df["polarity"].std():.4f}')

In [ ]:
# Sample labeled messages
print('Sample Labeled Messages:')
print('=' * 80)
for sentiment in ['Positive', 'Negative', 'Neutral']:
    sample = df[df['sentiment'] == sentiment].iloc[0]
    print(f'\n[{sentiment}] (Polarity: {sample["polarity"]:.3f})')
    print(f'  Subject: {sample["Subject"]}')
    body_preview = str(sample['body'])[:150].replace('\n', ' ')
    print(f'  Body: {body_preview}...')

In [ ]:
# Save labeled dataset
df.to_csv('test_labeled.csv', index=False)
print('Labeled dataset saved to: test_labeled.csv')

**Observation:** The sentiment labeling reveals the distribution of employee message sentiments. Most messages tend to fall into Neutral or Positive categories, which is expected for professional workplace communication. The negative messages represent a smaller but important subset for identifying engagement issues.

---
## Task 2: Exploratory Data Analysis (EDA)

In this section, we explore the dataset to understand its structure, distribution, and key patterns.

### 2.1 Sentiment Distribution

In [ ]:
# Sentiment Distribution - Bar Chart and Pie Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Positive': '#2ecc71', 'Neutral': '#3498db', 'Negative': '#e74c3c'}
counts = df['sentiment'].value_counts()

# Bar chart
bars = axes[0].bar(counts.index, counts.values,
                   color=[colors.get(x, '#95a5a6') for x in counts.index],
                   edgecolor='white', linewidth=1.5)
axes[0].set_title('Sentiment Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Number of Messages')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=[colors.get(x, '#95a5a6') for x in counts.index],
            startangle=140, shadow=True, explode=[0.05]*len(counts))
axes[1].set_title('Sentiment Distribution (%)', fontsize=14, fontweight='bold')

fig.suptitle('Overall Sentiment Distribution of Employee Messages',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualizations/01_sentiment_distribution.png', bbox_inches='tight', facecolor='white')
plt.show()

**Observation:** The bar chart and pie chart show the overall distribution of sentiments across all employee messages. This gives us a high-level view of the emotional tone of workplace communications.

### 2.2 Sentiment Trends Over Time

In [ ]:
# Sentiment trends over time
df['year_month'] = df['date'].dt.to_period('M')
monthly_sentiment = df.groupby(['year_month', 'sentiment']).size().unstack(fill_value=0)
monthly_sentiment.index = monthly_sentiment.index.astype(str)

fig, ax = plt.subplots(figsize=(16, 6))
for col in ['Positive', 'Neutral', 'Negative']:
    if col in monthly_sentiment.columns:
        ax.plot(monthly_sentiment.index, monthly_sentiment[col],
                label=col, color=colors[col], marker='o', linewidth=2, markersize=4)

ax.set_title('Sentiment Trends Over Time (Monthly)', fontsize=16, fontweight='bold')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Number of Messages', fontsize=12)
ax.legend(fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('visualizations/02_sentiment_over_time.png', bbox_inches='tight', facecolor='white')
plt.show()

**Observation:** The time-series plot reveals how sentiment patterns evolve over time. We can identify periods of increased negativity or positivity, which could correlate with company events, seasonal patterns, or organizational changes.

### 2.3 Polarity Score Distribution

In [ ]:
# Polarity distribution histogram
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df['polarity'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='Neutral (0)')
ax.axvline(x=df['polarity'].mean(), color='green', linestyle='--', linewidth=1.5,
           label=f'Mean ({df["polarity"].mean():.3f})')
ax.set_title('Distribution of Polarity Scores', fontsize=16, fontweight='bold')
ax.set_xlabel('Polarity Score', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('visualizations/03_polarity_distribution.png', bbox_inches='tight', facecolor='white')
plt.show()

**Observation:** The polarity score histogram shows the continuous distribution of sentiment. A large spike at 0 indicates many neutral messages. The spread of positive and negative scores shows the range of emotional expression in workplace emails.

### 2.4 Employee Activity Analysis

In [ ]:
# Top senders
msg_counts = df['from'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(range(len(msg_counts)), msg_counts.values, color='#3498db',
        edgecolor='white', linewidth=1)
ax.set_yticks(range(len(msg_counts)))
ax.set_yticklabels([e.split('@')[0] for e in msg_counts.index], fontsize=10)
ax.set_xlabel('Number of Messages', fontsize=12)
ax.set_title('Top 15 Employees by Message Count', fontsize=16, fontweight='bold')
ax.invert_yaxis()
for i, val in enumerate(msg_counts.values):
    ax.text(val + 5, i, str(val), va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/04_messages_per_employee.png', bbox_inches='tight', facecolor='white')
plt.show()

**Observation:** There is significant variation in message volume across employees. Some employees are highly active communicators while others send very few messages. This disparity should be considered when interpreting sentiment scores.

### 2.5 Sentiment Breakdown by Employee

In [ ]:
# Sentiment by top employees - Stacked bar
top_employees = df['from'].value_counts().head(10).index
emp_sentiment = df[df['from'].isin(top_employees)].groupby(['from', 'sentiment']).size().unstack(fill_value=0)
emp_sentiment = emp_sentiment.loc[top_employees]
emp_sentiment.index = [e.split('@')[0] for e in emp_sentiment.index]

fig, ax = plt.subplots(figsize=(14, 6))
bottom = np.zeros(len(emp_sentiment))
for sentiment in ['Positive', 'Neutral', 'Negative']:
    if sentiment in emp_sentiment.columns:
        vals = emp_sentiment[sentiment].values
        ax.bar(emp_sentiment.index, vals, bottom=bottom,
               label=sentiment, color=colors[sentiment], edgecolor='white')
        bottom += vals

ax.set_title('Sentiment Breakdown by Top 10 Employees', fontsize=16, fontweight='bold')
ax.set_xlabel('Employee', fontsize=12)
ax.set_ylabel('Number of Messages', fontsize=12)
ax.legend(fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('visualizations/05_sentiment_by_employee.png', bbox_inches='tight', facecolor='white')
plt.show()

**Observation:** The stacked bar chart reveals each employee's sentiment composition. Some employees have a higher proportion of positive messages while others lean more neutral or negative. This helps identify individual communication patterns.

### 2.6 Day of Week Patterns

In [ ]:
# Messages by day of week
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['day_of_week'] = df['date'].dt.dayofweek
day_counts = df['day_of_week'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#3498db'] * 5 + ['#e74c3c'] * 2
bars = ax.bar([day_names[i] for i in day_counts.index], day_counts.values,
              color=bar_colors, edgecolor='white', linewidth=1.5)
ax.set_title('Messages by Day of Week', fontsize=16, fontweight='bold')
ax.set_xlabel('Day of Week', fontsize=12)
ax.set_ylabel('Number of Messages', fontsize=12)
for bar, val in zip(bars, day_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/06_day_of_week.png', bbox_inches='tight', facecolor='white')
plt.show()

**Observation:** Message volume varies by day of the week. Weekdays (blue) generally see more messages than weekends (red). This follows expected patterns of workplace email activity.

### 2.7 Message Length by Sentiment

In [ ]:
# Message length by sentiment
df['message_length'] = df['full_text'].str.len()

# Clip outliers for better visualization
q99 = df['message_length'].quantile(0.99)
df_plot = df[df['message_length'] <= q99]

fig, ax = plt.subplots(figsize=(10, 6))
palette = [colors['Negative'], colors['Neutral'], colors['Positive']]
sns.boxplot(data=df_plot, x='sentiment', y='message_length',
            order=['Negative', 'Neutral', 'Positive'],
            palette=palette, ax=ax)
ax.set_title('Message Length by Sentiment', fontsize=16, fontweight='bold')
ax.set_xlabel('Sentiment', fontsize=12)
ax.set_ylabel('Message Length (characters)', fontsize=12)
plt.tight_layout()
plt.savefig('visualizations/07_message_length_by_sentiment.png', bbox_inches='tight', facecolor='white')
plt.show()

**Observation:** There are differences in message length across sentiment categories. This suggests that the way employees express themselves (long vs. short messages) may correlate with the sentiment of their communication.

### 2.8 Word Clouds by Sentiment

In [ ]:
# Word clouds for each sentiment
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sentiments = ['Positive', 'Neutral', 'Negative']
color_maps = ['Greens', 'Blues', 'Reds']

custom_stopwords = set(['enron', 'com', 'will', 'please', 'us', 'one', 'would',
                        'also', 'may', 'know', 'new', 'said', 'thank', 'thanks'])

for ax, sentiment, cmap in zip(axes, sentiments, color_maps):
    text = ' '.join(df[df['sentiment'] == sentiment]['full_text'].dropna().astype(str))
    if text.strip():
        wc = WordCloud(width=600, height=400, background_color='white',
                      colormap=cmap, max_words=100, stopwords=custom_stopwords)
        wc.generate(text)
        ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{sentiment} Messages', fontsize=14, fontweight='bold')
    ax.axis('off')

fig.suptitle('Word Clouds by Sentiment Category', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualizations/08_wordclouds.png', bbox_inches='tight', facecolor='white')
plt.show()

**Observation:** Word clouds highlight the most frequently used words in each sentiment category. Positive messages tend to use words like 'good', 'great', 'hope', while negative messages contain words with more critical or frustrated tones. Neutral messages are dominated by business-related terminology.

### 2.9 EDA Summary Statistics

In [ ]:
# EDA Summary
print('=' * 60)
print('  EXPLORATORY DATA ANALYSIS - SUMMARY')
print('=' * 60)
print(f'\nTotal messages: {len(df)}')
print(f'Unique employees: {df["from"].nunique()}')
print(f'Date range: {df["date"].min().strftime("%Y-%m-%d")} to {df["date"].max().strftime("%Y-%m-%d")}')
print(f'\nSentiment Distribution:')
for s in ['Positive', 'Neutral', 'Negative']:
    c = (df['sentiment'] == s).sum()
    print(f'  {s}: {c} ({c/len(df)*100:.1f}%)')
print(f'\nAverage message length: {df["message_length"].mean():.0f} characters')
print(f'Average polarity: {df["polarity"].mean():.4f}')
print(f'\nKey EDA Insights:')
print('  1. The dataset contains Enron employee email messages')
print('  2. Most messages are Neutral or Positive in sentiment')
print('  3. Some employees are significantly more active than others')
print('  4. Message activity is higher on weekdays vs weekends')
print('  5. Message length varies across sentiment categories')

---
## Task 3: Monthly Employee Sentiment Scoring

### Scoring Method
For each employee, each message receives a score:
- **Positive message:** +1
- **Negative message:** -1
- **Neutral message:** 0 (no effect)

Scores are aggregated on a monthly basis and **reset at the start of each new month**.

In [ ]:
# Calculate sentiment scores
def calculate_sentiment_score(sentiment):
    if sentiment == 'Positive':
        return 1
    elif sentiment == 'Negative':
        return -1
    else:
        return 0

df['sentiment_score'] = df['sentiment'].apply(calculate_sentiment_score)
df['year_month'] = df['date'].dt.to_period('M')

# Compute monthly scores
monthly_scores = df.groupby(['from', 'year_month']).agg(
    monthly_score=('sentiment_score', 'sum'),
    positive_count=('sentiment_score', lambda x: (x == 1).sum()),
    negative_count=('sentiment_score', lambda x: (x == -1).sum()),
    neutral_count=('sentiment_score', lambda x: (x == 0).sum()),
    total_messages=('sentiment_score', 'count')
).reset_index()

print(f'Monthly scores computed for {monthly_scores["from"].nunique()} employees')
print(f'Across {monthly_scores["year_month"].nunique()} unique months')
print(f'\nMonthly Score Statistics:')
print(f'  Mean:  {monthly_scores["monthly_score"].mean():.2f}')
print(f'  Std:   {monthly_scores["monthly_score"].std():.2f}')
print(f'  Max:   {monthly_scores["monthly_score"].max()}')
print(f'  Min:   {monthly_scores["monthly_score"].min()}')

In [ ]:
# Display sample monthly scores
print('Sample Monthly Scores (first 15 records):')
monthly_scores.head(15)

In [ ]:
# Monthly Scores Heatmap
top_emps = monthly_scores.groupby('from')['total_messages'].sum().nlargest(15).index
filtered = monthly_scores[monthly_scores['from'].isin(top_emps)].copy()
pivot = filtered.pivot_table(index='from', columns='year_month', values='monthly_score', fill_value=0)
pivot.index = [e.split('@')[0] for e in pivot.index]
pivot.columns = pivot.columns.astype(str)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot, cmap='RdYlGn', center=0, annot=True, fmt='.0f',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Monthly Score'})
ax.set_title('Monthly Sentiment Scores Heatmap (Top 15 Employees)',
             fontsize=16, fontweight='bold')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Employee', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('visualizations/09_monthly_scores_heatmap.png', bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Save monthly scores
monthly_scores.to_csv('monthly_scores.csv', index=False)
print('Monthly scores saved to: monthly_scores.csv')

**Observation:** The heatmap provides a clear visual overview of how each employee's sentiment score fluctuates across months. Green cells indicate positive months, red cells indicate negative months, and yellow represents neutral periods. This helps identify consistent patterns of positive or negative engagement.

---
## Task 4: Employee Ranking

### Ranking Method
For each month, we generate:
1. **Top 3 Positive Employees** - Highest monthly score (ties broken alphabetically)
2. **Top 3 Negative Employees** - Lowest monthly score (ties broken alphabetically)

In [ ]:
# Generate rankings for each month
top_positive_list = []
top_negative_list = []

for month in monthly_scores['year_month'].unique():
    month_data = monthly_scores[monthly_scores['year_month'] == month].copy()
    
    # Top 3 Positive: sort descending by score, then alphabetically
    sorted_pos = month_data.sort_values(by=['monthly_score', 'from'], ascending=[False, True])
    top_3_pos = sorted_pos.head(3).copy()
    top_3_pos['rank'] = range(1, len(top_3_pos) + 1)
    top_positive_list.append(top_3_pos)
    
    # Top 3 Negative: sort ascending by score, then alphabetically
    sorted_neg = month_data.sort_values(by=['monthly_score', 'from'], ascending=[True, True])
    top_3_neg = sorted_neg.head(3).copy()
    top_3_neg['rank'] = range(1, len(top_3_neg) + 1)
    top_negative_list.append(top_3_neg)

top_positive = pd.concat(top_positive_list, ignore_index=True)
top_negative = pd.concat(top_negative_list, ignore_index=True)

print('Employee rankings computed for all months!')

In [ ]:
# Display rankings for each month
for month in sorted(monthly_scores['year_month'].unique()):
    print(f'\n{"="*60}')
    print(f'  Month: {month}')
    print(f'{"="*60}')
    
    pos_month = top_positive[top_positive['year_month'] == month]
    neg_month = top_negative[top_negative['year_month'] == month]
    
    print('  [+] Top 3 POSITIVE Employees:')
    for _, row in pos_month.iterrows():
        print(f'      #{int(row["rank"])}  {row["from"]:<40} Score: {row["monthly_score"]:>3}')
    
    print('  [-] Top 3 NEGATIVE Employees:')
    for _, row in neg_month.iterrows():
        print(f'      #{int(row["rank"])}  {row["from"]:<40} Score: {row["monthly_score"]:>3}')

In [ ]:
# Employee Ranking Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Aggregate across all months for overall ranking
pos_agg = top_positive.groupby('from')['monthly_score'].mean().nlargest(5)
neg_agg = top_negative.groupby('from')['monthly_score'].mean().nsmallest(5)

# Top Positive
labels_pos = [e.split('@')[0] for e in pos_agg.index]
axes[0].barh(range(len(pos_agg)), pos_agg.values, color='#2ecc71', edgecolor='white')
axes[0].set_yticks(range(len(pos_agg)))
axes[0].set_yticklabels(labels_pos, fontsize=10)
axes[0].set_title('Top Positive Employees (Avg Monthly Score)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Average Monthly Score')
axes[0].invert_yaxis()

# Top Negative
labels_neg = [e.split('@')[0] for e in neg_agg.index]
axes[1].barh(range(len(neg_agg)), neg_agg.values, color='#e74c3c', edgecolor='white')
axes[1].set_yticks(range(len(neg_agg)))
axes[1].set_yticklabels(labels_neg, fontsize=10)
axes[1].set_title('Top Negative Employees (Avg Monthly Score)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Average Monthly Score')
axes[1].invert_yaxis()

fig.suptitle('Employee Rankings by Sentiment Score', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualizations/10_employee_rankings.png', bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Save rankings
top_positive.to_csv('top_positive_employees.csv', index=False)
top_negative.to_csv('top_negative_employees.csv', index=False)
print('Rankings saved to: top_positive_employees.csv, top_negative_employees.csv')

**Observation:** The ranking system clearly identifies the most positive and most negative employees for each month. Employees who consistently appear in the top positive list demonstrate strong engagement, while those in the negative list may require attention from management.

---
## Task 5: Flight Risk Identification

### Flight Risk Criteria
An employee is flagged as a **flight risk** if they have sent **4 or more negative messages** within any **rolling 30-day window** (irrespective of calendar month boundaries).

### Approach
- Filter only negative messages per employee
- Sort by date
- Use a sliding window approach to check all possible 30-day windows
- Flag employees where any window contains 4+ negative messages

In [ ]:
# Flight Risk Identification
negative_msgs = df[df['sentiment'] == 'Negative'].copy()
negative_msgs = negative_msgs.sort_values(['from', 'date'])

flight_risk_employees = []

for employee in negative_msgs['from'].unique():
    emp_negatives = negative_msgs[negative_msgs['from'] == employee].sort_values('date').reset_index(drop=True)
    
    if len(emp_negatives) < 4:
        continue
    
    # Rolling 30-day window check
    dates = emp_negatives['date'].values
    
    for i in range(len(dates)):
        window_start = dates[i]
        window_end = dates[i] + np.timedelta64(30, 'D')
        count = ((dates >= window_start) & (dates <= window_end)).sum()
        
        if count >= 4:
            flight_risk_employees.append({
                'employee': employee,
                'total_negative_messages': len(emp_negatives),
                'max_negatives_in_30_days': count,
                'risk_window_start': pd.Timestamp(window_start),
                'risk_window_end': pd.Timestamp(window_end)
            })
            break  # Found risk, move to next employee

flight_risks = pd.DataFrame(flight_risk_employees)
print(f'{len(flight_risks)} employees identified as FLIGHT RISKS')

In [ ]:
# Display flight risk employees
if flight_risks.empty:
    print('No employees were identified as flight risks.')
else:
    print('Flight Risk Employees:')
    print('=' * 70)
    for _, row in flight_risks.iterrows():
        print(f'\n  [!] {row["employee"]}')
        print(f'      Total negative messages: {row["total_negative_messages"]}')
        print(f'      Max negatives in 30-day window: {row["max_negatives_in_30_days"]}')
        print(f'      Risk window: {row["risk_window_start"].strftime("%Y-%m-%d")} to {row["risk_window_end"].strftime("%Y-%m-%d")}')

In [ ]:
# Flight Risk Visualization
if not flight_risks.empty:
    fr = flight_risks.sort_values('max_negatives_in_30_days', ascending=False)
    
    fig, ax = plt.subplots(figsize=(14, max(5, len(fr) * 0.5)))
    labels = [e.split('@')[0] for e in fr['employee']]
    colors_fr = ['#e74c3c' if x >= 6 else '#e67e22' if x >= 5 else '#f1c40f'
                 for x in fr['max_negatives_in_30_days']]
    
    ax.barh(range(len(fr)), fr['max_negatives_in_30_days'].values,
            color=colors_fr, edgecolor='white', linewidth=1)
    ax.set_yticks(range(len(fr)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Max Negative Messages in 30-Day Window', fontsize=12)
    ax.set_title('Flight Risk Employees (>=4 Negative Messages in 30 Days)',
                 fontsize=16, fontweight='bold')
    ax.axvline(x=4, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Threshold (4)')
    ax.legend(fontsize=11)
    ax.invert_yaxis()
    for i, val in enumerate(fr['max_negatives_in_30_days'].values):
        ax.text(val + 0.1, i, str(val), va='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig('visualizations/11_flight_risk_employees.png', bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print('No flight risk visualization to generate.')

In [ ]:
# Save flight risks
if not flight_risks.empty:
    flight_risks.to_csv('flight_risk_employees.csv', index=False)
    print('Flight risk list saved to: flight_risk_employees.csv')
else:
    print('No flight risk employees to save.')

**Observation:** The flight risk identification uses a robust rolling 30-day window approach that is independent of calendar month boundaries. Employees flagged here have shown a concentrated pattern of negative communication, which could indicate disengagement, frustration, or potential intent to leave the organization.

---
## Task 6: Linear Regression Predictive Model

### Model Approach
We build a linear regression model to analyze sentiment trends and predict polarity scores using engineered features.

### Independent Variables (Features):
| Feature | Description |
|---------|-------------|
| message_length | Total character count |
| word_count | Number of words |
| subject_length | Subject line length |
| has_subject | Has a subject (0/1) |
| day_of_week | Day (0=Mon, 6=Sun) |
| is_weekend | Weekend flag (0/1) |
| month | Month (1-12) |
| exclamation_count | Count of '!' |
| question_count | Count of '?' |
| caps_ratio | Uppercase ratio |

### Dependent Variable (Target):
- `polarity` (continuous, -1 to +1)

### Library:
- scikit-learn (LinearRegression with StandardScaler)

In [ ]:
# Feature Engineering
df['word_count'] = df['full_text'].str.split().str.len()
df['subject_length'] = df['Subject'].fillna('').str.len()
df['has_subject'] = (df['Subject'].fillna('') != '').astype(int)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['month'] = df['date'].dt.month
df['exclamation_count'] = df['full_text'].str.count('!')
df['question_count'] = df['full_text'].str.count(r'\?')

# Caps ratio
def caps_ratio(text):
    if pd.isna(text) or len(text) == 0:
        return 0.0
    alpha_chars = [c for c in text if c.isalpha()]
    if len(alpha_chars) == 0:
        return 0.0
    return sum(1 for c in alpha_chars if c.isupper()) / len(alpha_chars)

df['caps_ratio'] = df['full_text'].apply(caps_ratio)

# Prepare features and target
feature_cols = ['message_length', 'word_count', 'subject_length',
                'has_subject', 'day_of_week', 'is_weekend', 'month',
                'exclamation_count', 'question_count', 'caps_ratio']

X = df[feature_cols].fillna(0)
y = df['polarity']

print(f'Features: {feature_cols}')
print(f'Total samples: {len(X)}')
print(f'\nFeature statistics:')
X.describe()

In [ ]:
# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {len(X_train)} samples')
print(f'Testing set:  {len(X_test)} samples')

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train Linear Regression Model
model = LinearRegression()
model.fit(X_train_scaled, y_train)
print('Linear Regression model trained successfully!')

# Predictions
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

# Evaluation Metrics
print('\nModel Performance Metrics:')
print('=' * 50)
print(f'  Training R2 Score:  {r2_score(y_train, y_pred_train):.4f}')
print(f'  Testing R2 Score:   {r2_score(y_test, y_pred_test):.4f}')
print(f'  Training RMSE:      {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}')
print(f'  Testing RMSE:       {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}')
print(f'  Training MAE:       {mean_absolute_error(y_train, y_pred_train):.4f}')
print(f'  Testing MAE:        {mean_absolute_error(y_test, y_pred_test):.4f}')

In [ ]:
# Feature Coefficients
print('Feature Coefficients (Scaled):')
print('=' * 50)
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)

for _, row in coef_df.iterrows():
    direction = '+' if row['Coefficient'] > 0 else '-'
    print(f'  {direction} {row["Feature"]:<25}: {row["Coefficient"]:>10.6f}')

print(f'\n  Intercept: {model.intercept_:.6f}')

coef_df

In [ ]:
# Regression Results Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_test, alpha=0.3, color='#3498db', s=10)
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Polarity', fontsize=12)
axes[0].set_ylabel('Predicted Polarity', fontsize=12)
axes[0].set_title('Actual vs Predicted Polarity Scores', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

# Feature Importance
importance = pd.Series(model.coef_, index=feature_cols).sort_values()
colors_fi = ['#e74c3c' if v < 0 else '#2ecc71' for v in importance.values]
axes[1].barh(range(len(importance)), importance.values, color=colors_fi, edgecolor='white')
axes[1].set_yticks(range(len(importance)))
axes[1].set_yticklabels(importance.index, fontsize=10)
axes[1].set_xlabel('Coefficient Value', fontsize=12)
axes[1].set_title('Feature Importance (Linear Regression Coefficients)',
                  fontsize=14, fontweight='bold')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

fig.suptitle('Linear Regression Model Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualizations/12_regression_results.png', bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Residual Diagnostic Plots
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual histogram
axes[0].hist(residuals, bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Residual Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Frequency')

# Residual scatter
axes[1].scatter(y_pred_test, residuals, alpha=0.3, color='#e74c3c', s=10)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_title('Residuals vs Predicted Values', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Polarity')
axes[1].set_ylabel('Residual')

fig.suptitle('Model Diagnostic Plots', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualizations/13_residuals.png', bbox_inches='tight', facecolor='white')
plt.show()

### Model Interpretation

**Key Findings:**
- The linear regression model captures some variance in sentiment polarity, though text sentiment is inherently difficult to predict from metadata features alone.
- Features like exclamation count, message length, and caps ratio show meaningful correlations with sentiment polarity.
- The R-squared score indicates the proportion of variance explained by the model.
- A relatively low R-squared is expected since sentiment is primarily determined by actual text content, not structural features.
- The model serves as a baseline for understanding which structural attributes correlate with sentiment.

---
## Summary of Key Findings

In [ ]:
# Final Summary
print('=' * 70)
print('  EMPLOYEE SENTIMENT ANALYSIS - FINAL SUMMARY')
print('=' * 70)

print('\n[1] SENTIMENT OVERVIEW:')
for sentiment, count in sentiment_counts.items():
    pct = count / len(df) * 100
    print(f'    {sentiment}: {count} ({pct:.1f}%)')

print('\n[2] TOP POSITIVE EMPLOYEES (Overall):')
overall_pos = monthly_scores.groupby('from')['monthly_score'].sum().nlargest(3)
for i, (emp, score) in enumerate(overall_pos.items(), 1):
    print(f'    #{i} {emp} (Total Score: {score})')

print('\n[3] TOP NEGATIVE EMPLOYEES (Overall):')
overall_neg = monthly_scores.groupby('from')['monthly_score'].sum().nsmallest(3)
for i, (emp, score) in enumerate(overall_neg.items(), 1):
    print(f'    #{i} {emp} (Total Score: {score})')

print('\n[4] FLIGHT RISK EMPLOYEES:')
if flight_risks.empty:
    print('    No employees flagged as flight risks.')
else:
    for _, row in flight_risks.iterrows():
        print(f'    [!] {row["employee"]} ({row["max_negatives_in_30_days"]} negatives in 30 days)')

print('\n[5] MODEL PERFORMANCE:')
print(f'    R2 Score (Test): {r2_score(y_test, y_pred_test):.4f}')
print(f'    RMSE (Test):     {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}')
print(f'    MAE (Test):      {mean_absolute_error(y_test, y_pred_test):.4f}')

print('\n[6] KEY RECOMMENDATIONS:')
print('    - Monitor flight risk employees closely for signs of disengagement')
print('    - Investigate root causes of negative sentiment patterns')
print('    - Consider implementing regular sentiment check-ins')
print('    - Use sentiment trends to inform retention strategies')

print('\n' + '=' * 70)
print('  PROJECT COMPLETE')
print('=' * 70)

---
## Output Files Generated

| File | Description |
|------|-------------|
| `test_labeled.csv` | Dataset with sentiment labels |
| `monthly_scores.csv` | Monthly sentiment scores per employee |
| `top_positive_employees.csv` | Top 3 positive employees per month |
| `top_negative_employees.csv` | Top 3 negative employees per month |
| `flight_risk_employees.csv` | Employees flagged as flight risks |
| `visualizations/` | All generated charts and plots (13 visualizations) |